# Fitness app users — 01 Data understanding and validation

Eric Gomez · November 2026

A fitness-tracking app's product team supplied one year of user records: each user's gender, age, location
type, activity level, number of app sessions, distance travelled and calories burned. The team wants to use
these data to build personalised workout recommendations.

This notebook profiles the raw file and checks it record by record. Each problem is given a classification
and a decision, with the evidence behind it. Where the evidence leaves room for doubt, the alternative is
carried into the sensitivity analysis in `04`.

| Notebook | Contents |
|---|---|
| `01` Data understanding | Profile, validation, a decision for each problem |
| `02` EDA | Cleaning, validation and management matrix, patterns, figures |
| `03` Analysis | Pre-stated hypotheses on demographic differences, with effect sizes and equivalence tests |
| `04` Modelling | Calorie regression, activity-tier classification, user segmentation; leakage and sensitivity checks |
| `05` Ethics and culture | Re-identification risk, differential privacy, subgroup performance, label validity; the dashboard |

In [1]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

## The question

Which patterns of engagement do these records support, which of them can be predicted reliably, and what
can the data not answer? The answer is for the product team designing the recommendation feature: it decides
which signals the feature can rely on, which data the app should collect next, and which it should not.

Four things the file does not record could change the answers:

- **A data dictionary.** No units or period are given for *Calories Burned*, and the rule that assigns
  *Activity Level* is not stated (it is reconstructed below).
- **Time.** There are no dates or timestamps, so seasonality, trends and churn cannot be studied.
- **Session detail.** Duration, heart rate and workout type are not recorded, so effort per session can only be
  inferred from distance and calories.
- **App performance.** There are no crash, latency or error logs, so "app performance" can only be read through
  engagement.

In [2]:
from src.data.loading import list_raw, load_raw
from src.data.quality import cardinality_report, profile, schema_report

DATASET = 'dataset for assignment 2-0426.csv'
print('files in data/raw/:', list_raw())
raw = load_raw(DATASET)
profile(raw)
display(schema_report(raw))
display(cardinality_report(raw))

files in data/raw/: ['dataset for assignment 2-0426.csv']
rows: 5,299   columns: 8
memory: 1,064.4 KB

dtypes:
int64      3
str        3
float64    2


,column,dtype,non_null,missing,missing_pct,unique,example
0,User ID,int64,5299,0,0.00,5299,1
1,Gender,str,5299,0,0.00,2,Female
2,Age,int64,5299,0,0.00,42,22
3,Activity Level,str,5299,0,0.00,3,Active
4,Location,str,5299,0,0.00,3,Suburban
5,App Sessions,float64,5298,1,0.02,150,151.0
6,Distance Travelled (km),float64,5298,1,0.02,171,140.0
7,Calories Burned,int64,5299,0,0.00,800,728


,column,unique,unique_ratio,note
0,User ID,5299,1.000,
7,Calories Burned,800,0.151,
6,Distance Travelled (km),171,0.032,
5,App Sessions,150,0.028,
2,Age,42,0.008,
3,Activity Level,3,0.001,
4,Location,3,0.001,
1,Gender,2,0.000,


### Interpretation

- **Size and shape.** 5,299 records and 8 columns, one record per user. The memory footprint is about 1 MB,
  so every analysis below runs on the full file.
- **Types.** Three text columns (*Gender*, *Activity Level*, *Location*) and five numbers. *App Sessions* and
  *Distance Travelled (km)* load as decimals only because each has one blank cell (checked below).
- **Cardinality.** *User ID* is unique for every record: it is an identifier, not a feature. The three text
  columns have two or three levels each, with no spelling variants.

In [3]:
ids = raw['User ID']
print('User ID unique:', ids.is_unique, '| range', ids.min(), '-', ids.max(),
      '| consecutive:', bool((ids.diff().dropna() == 1).all()))
for col in ['Gender', 'Activity Level', 'Location']:
    print(f'{col}: {raw[col].value_counts().to_dict()}')

NUM = ['Age', 'App Sessions', 'Distance Travelled (km)', 'Calories Burned']
print('\nwhole numbers only:', {c: bool((raw[c].dropna() % 1 == 0).all()) for c in NUM})
raw[NUM].describe().round(1)

User ID unique: True | range 1 - 5299 | consecutive: True
Gender: {'Male': 2699, 'Female': 2600}
Activity Level: {'Moderate': 1834, 'Sedentary': 1742, 'Active': 1723}
Location: {'Rural': 1785, 'Suburban': 1776, 'Urban': 1738}

whole numbers only: {'Age': True, 'App Sessions': True, 'Distance Travelled (km)': True, 'Calories Burned': True}


,Age,App Sessions,Distance Travelled (km),Calories Burned
count,5299.0,5298.0,5298.0,5299.0
mean,38.4,124.2,92.7,432.1
std,12.2,42.8,38.7,187.8
min,18.0,50.0,10.0,102.0
25%,28.0,89.0,63.0,284.0
50%,39.0,124.0,89.0,406.0
75%,49.0,161.0,118.0,559.5
max,59.0,199.0,920.0,987.0


### Interpretation

- **Identifiers.** *User ID* runs from 1 to 5,299 with no gaps or repeats.
- **Categories are clean and balanced.** Gender is split 2,699 male and 2,600 female; each activity level and
  each location type holds about a third of the records. No category has a typo, a stray space or a
  different case.
- **Every number is a whole number.** Age runs from 18 to 59, App Sessions from 50 to 199 and Calories Burned
  from 102 to 987.
- **One value stands out.** The largest distance is 920 km, nearly eight times the 75th percentile (118 km).
  The smallest, 10 km, is also low for its row. Both are examined under *Consistency rules* below.
- **Units.** Distance is in kilometres. Calories Burned is most likely an annual total in kilocalories, but at
  102–987 kcal a year it is far below what real exercise burns (a single hour of brisk walking is a few hundred
  kcal). The values are therefore treated as relative measures: they rank users correctly, but they are not
  physiological totals.

In [4]:
from src.data.quality import missing_report

display(missing_report(raw))
raw[raw.isna().any(axis=1)]

complete rows: 5,297 of 5,299 (100.0%)


,missing,missing_pct
App Sessions,1,0.02
Distance Travelled (km),1,0.02


,User ID,Gender,Age,Activity Level,Location,App Sessions,Distance Travelled (km),Calories Burned
4906,4907,Female,44,Active,Rural,155.0,NaN,582
5258,5259,Female,33,Sedentary,Suburban,NaN,91.0,435


### Interpretation

- **Two blank cells, in two different records.** User 4907 has no distance and user 5259 has no session count.
- **Different stories.** User 5259 belongs to a block of copied records (see *Data-integrity errors*), so its
  blank is part of an altered copy rather than a real gap. User 4907 is an ordinary record with one value
  missing.

## Validation approach

Each problem found below is given one of four classifications. Each classification has a test and a default
action.

| Classification | Test | Default action |
|---|---|---|
| **Critical anomaly** | The value is physically impossible | Remove the value or the record; it cannot enter any analysis |
| **Data-integrity error** | The record is provably not an independent observation: a copy of a record above it, verbatim or altered | Remove it and keep the source |
| **Statistical outlier** | Extreme or inconsistent, physically possible, and not provably an error | Decide case by case, and test the alternative in the sensitivity analysis |
| **Missing data** | A blank cell | Keep the record; leave the cell blank unless a model needs it filled, and say how |

A record takes the first label that fits, in the order of the table.

In [5]:
from src.data.quality import duplicate_report

FIELDS = [c for c in raw.columns if c != 'User ID']      # everything that describes the user
duplicate_report(raw, subset=FIELDS)

def record_key(d):
    """One string per record over FIELDS; a blank cell becomes 'NA' so it can match."""
    return d[FIELDS].fillna(-1).astype(str).apply('|'.join, axis=1)

copy_mask = raw.duplicated(subset=FIELDS, keep='first')
first = raw[~copy_mask]
source_of = dict(zip(record_key(first), first['User ID']))
verbatim = raw[copy_mask].assign(source_id=lambda d: record_key(d).map(source_of))
verbatim['id_offset'] = verbatim['User ID'] - verbatim['source_id']
top_offset = verbatim['id_offset'].mode().iat[0]

print(f'\nverbatim copies: {len(verbatim)}')
print('copy IDs  :', verbatim['User ID'].min(), '-', verbatim['User ID'].max())
print('source IDs:', verbatim['source_id'].min(), '-', verbatim['source_id'].max())
print('times each source is copied:', verbatim['source_id'].value_counts().value_counts().sort_index().to_dict())
print(f"most common ID offset: {top_offset} ({(verbatim['id_offset'] == top_offset).mean():.0%} of copies)")
print('verbatim copies with an ID of 5000 or below:', int((verbatim['User ID'] <= 5000).sum()))

exact duplicate rows: 0
duplicate on ['Gender', 'Age', 'Activity Level', 'Location', 'App Sessions', 'Distance Travelled (km)', 'Calories Burned']: 272

verbatim copies: 272
copy IDs  : 5001 - 5299
source IDs: 3580 - 3855
times each source is copied: {1: 244, 2: 14}
most common ID offset: 1442 (92% of copies)
verbatim copies with an ID of 5000 or below: 0


### Interpretation

- **272 verbatim copies.** These records repeat an earlier record in every field except *User ID*. No two
  records with an ID of 5000 or below are copies of each other, so the copies sit only in IDs 5001–5299.
- **Taken from a narrow range.** Every copy repeats a user between 3580 and 3855. 244 of those users are copied
  once and 14 twice. 92% of the copies sit exactly 1,442 IDs after their source, the signature of a block that was
  pasted at the end of the file and re-numbered.
- **Why exact matches mean copies, not coincidence.** A record is a combination of two genders, 42 ages,
  three activity levels, three locations, 150 session counts, about 170 distances and 800 calorie values.
  Two independent users matching on all seven fields is vanishingly unlikely, and the matches here cluster in
  one ID range.

In [6]:
block = raw[raw['User ID'] > 5000]
others = block[~copy_mask.loc[block.index]]
print(f'records after ID 5000: {len(block)} = {len(block) - len(others)} verbatim copies + {len(others)} others')

base = raw[raw['User ID'] <= 5000].reset_index(drop=True)
lo_src, hi_src = verbatim['source_id'].min(), verbatim['source_id'].max()

def show(v):
    return 'blank' if pd.isna(v) else (f'{v:g}' if isinstance(v, float) else str(v))

def nearest_source(row, pool):
    """The pool record sharing the most FIELDS with `row`. A blank never matches.
    Ties go to a record inside the range the verbatim copies were taken from."""
    same = pd.DataFrame({f: pool[f].eq(row[f]) for f in FIELDS})
    n_same = same.sum(axis=1)
    hits = pool[n_same == n_same.max()]
    in_range = hits[hits['User ID'].between(lo_src, hi_src)]
    src = (in_range if len(in_range) else hits).iloc[0]
    changed = [f for f in FIELDS if not same.loc[src.name, f]]
    return src, int(n_same.max()), changed

rows = []
for _, r in others.iterrows():
    src, n_same, changed = nearest_source(r, base)
    rows.append({'user_id': r['User ID'], 'source_id': src['User ID'],
                 'fields_matching': f'{n_same} of 7', 'changed': ', '.join(changed),
                 'source_value': '; '.join(show(src[c]) for c in changed),
                 'copy_value': '; '.join(show(r[c]) for c in changed)})
altered = pd.DataFrame(rows)
print('sources inside the verbatim-copy source range:', int(altered['source_id'].between(lo_src, hi_src).sum()), 'of', len(altered))
print(altered['changed'].value_counts().to_string())
altered

records after ID 5000: 299 = 272 verbatim copies + 27 others
sources inside the verbatim-copy source range: 27 of 27
changed
Gender                    13
Location                   8
Gender, Location           2
App Sessions               1
Gender, App Sessions       1
Location, App Sessions     1
Calories Burned            1


,user_id,source_id,fields_matching,changed,source_value,copy_value
0,5010,3816,6 of 7,App Sessions,110,100
1,5018,3824,5 of 7,"Gender, App Sessions",Male; 100,Female; 90
2,5166,3724,6 of 7,Gender,Female,Male
3,5168,3726,6 of 7,Gender,Female,Male
4,5172,3730,6 of 7,Gender,Female,Male
5,5174,3732,6 of 7,Gender,Female,Male
6,5177,3735,6 of 7,Gender,Female,Male
7,5178,3736,6 of 7,Gender,Female,Male
8,5179,3737,6 of 7,Gender,Female,Male
9,5180,3738,6 of 7,Gender,Female,Male


### Interpretation

- **The block after ID 5000 adds no new user.** Its 299 records are the 272 verbatim copies plus 27 records
  that each match a user in 3580–3855 on five or six of the seven fields, always a user from the range the
  verbatim copies came from.
- **What was changed in the 27 altered copies:**
  - **gender flipped** in 16 records, 15 of them from Female to Male;
  - **location changed** in 11 records, mostly to Suburban;
  - **sessions changed** in 3: 110 → 100, 100 → 90, and one blanked (user 5259, the missing value above);
  - **calories changed** in 1: 132 → 332 (user 5279).
- **Why it matters.** Kept, these records would add 27 users who do not exist. Because most changes push in
  one direction, they would also distort the very comparisons the recommendation feature depends on: the
  gender balance shifts by 14 records towards Male, and 7 of the 11 location changes move a user to Suburban. User 5279's altered calories also
  break the calorie rule found below (332 kcal from 53 sessions is 6.3 kcal per session, outside the 2–5 band of
  every genuine record).

## Data-integrity errors

| Records | Problem | Decision | Evidence |
|---|---|---|---|
| 272 records in IDs 5001–5299 | Verbatim copies | **remove the copy, keep the source** | Identical in all seven fields to a user in 3580–3855; 92% at an ID offset of exactly 1,442 |
| 27 records in IDs 5001–5299 | Altered copies (gender, location, sessions or calories changed; one session count blanked) | **remove the copy, keep the source** | Each matches a user in the same source range on five or six of seven fields (table above) |

Together these are the whole block after ID 5000 (299 records, 5.6% of the file). The analysis keeps the 5,000
users with IDs 1–5000.

## Consistency rules

One-column screens judge a value against its column. A record can also be checked against itself: distance and calories should scale with the number of sessions.

In [7]:
b = base.copy()
b['km_per_session'] = b['Distance Travelled (km)'] / b['App Sessions']
b['kcal_per_session'] = b['Calories Burned'] / b['App Sessions']

r = b['km_per_session'].dropna().sort_values()
print('lowest km per session :', r.head(4).round(3).tolist())
print('highest km per session:', r.tail(4).round(3).tolist())
print('kcal per session range:', round(b['kcal_per_session'].min(), 3), '-', round(b['kcal_per_session'].max(), 3))

outside = b[(b['km_per_session'] < 0.45) | (b['km_per_session'] > 1.05)].copy()
outside['distance_if_x10_slip'] = np.where(outside['km_per_session'] > 1,
                                           outside['Distance Travelled (km)'] / 10,
                                           outside['Distance Travelled (km)'] * 10)
outside['km_per_session_then'] = outside['distance_if_x10_slip'] / outside['App Sessions']
outside.round(3)

lowest km per session : [0.084, 0.491, 0.491, 0.491]
highest km per session: [0.995, 0.995, 0.995, 5.055]
kcal per session range: 2.0 - 4.995


,User ID,Gender,Age,Activity Level,Location,App Sessions,Distance Travelled (km),Calories Burned,km_per_session,kcal_per_session,distance_if_x10_slip,km_per_session_then
1998,1999,Male,45,Active,Urban,182.0,920.0,909,5.055,4.995,92.0,0.505
2066,2067,Male,43,Moderate,Urban,119.0,10.0,245,0.084,2.059,100.0,0.840


### Interpretation

- **Distance per session follows a tight rule.** Across the 4,999 users with both values, distance divided by
  sessions sits between 0.49 and 0.995 km for every user but two. Calories per session sits between 2.0 and 5.0
  for all 5,000.
- **The two exceptions are a factor of ten out.**
  - **User 1999** recorded 920 km from 182 sessions, 5.05 km per session. At 92 km the ratio would be 0.51,
    inside the band.
  - **User 2067** recorded 10 km from 119 sessions, 0.08 km per session. At 100 km it would be 0.84.
  - Both look like a slipped digit (an extra or a dropped zero). The rest of each record is consistent:
    both calorie values sit inside the 2–5 kcal-per-session band.

In [8]:
display(b.groupby('Activity Level')['App Sessions'].agg(['min', 'max', 'count']))
rule = pd.cut(b['App Sessions'], [49, 99, 149, 199], labels=['Sedentary', 'Moderate', 'Active'])
agree = int((rule.astype(str) == b['Activity Level']).sum())
print(f"records whose Activity Level equals the band of their App Sessions: {agree} of {int(b['App Sessions'].notna().sum())}")

,min,max,count
Activity Level,,,
Active,150.0,199.0,1632
Moderate,100.0,149.0,1734
Sedentary,50.0,99.0,1634


records whose Activity Level equals the band of their App Sessions: 5000 of 5000


### Interpretation

- **Activity Level is a banding of App Sessions, without exception.** Sedentary is 50–99 sessions, Moderate
  100–149 and Active 150–199, with no overlap: all 5,000 records follow the rule.
- **It is a definition, not a finding.** *Activity Level* is not an independent measure of how active a user
  is. It measures how often the app was opened. Two consequences follow:
  - predicting *Activity Level* from *App Sessions* would score 100% and teach nothing (target leakage,
    `04`);
  - a user who exercises without opening the app is labelled Sedentary, whatever their real activity (`05`).

In [9]:
from src.data.quality import outlier_report

# Univariate screens, for comparison with the per-session rule above.
outlier_report(base[NUM])

Values flagged. Flagged is not the same as wrong -- decide and record why.


,column,n,iqr_flagged,z_flagged,lower_fence,upper_fence,min,max
0,Age,5000,0,0,-3.5,80.5,18.0,59.0
1,App Sessions,5000,0,0,-19.0,269.0,50.0,199.0
2,Distance Travelled (km),4999,1,1,-19.5,200.5,10.0,920.0
3,Calories Burned,5000,3,0,-127.5,972.5,102.0,987.0


### Interpretation

- **Univariate screens get the two errors half right.** The IQR fence and the z-score both flag the 920 km
  distance, but neither flags the 10 km distance: 10 km is low, yet inside the range of the column.
- **They also raise false alarms.** The IQR fence flags three calorie values (975, 980 and 987 kcal). All three
  belong to users with about 198 sessions, which is exactly where the highest calories should be.
- **The relational rule is the better screen.** Distance *per session* catches both errors and raises no false
  alarm. A rule that compares fields within a record catches what a one-column screen misses.

## Critical anomalies and statistical outliers

| Record | Value | Classification | Decision | Evidence |
|---|---|---|---|---|
| User 1999 | distance **920 km** | Statistical outlier | **set the distance to blank, keep the record** | Possible in a year, but 5.05 km per session against 0.49–0.995 for every other user; 92 km would fit. Sessions, calories and demographics are consistent |
| User 2067 | distance **10 km** | Statistical outlier | **set the distance to blank, keep the record** | 0.08 km per session; 100 km would fit. Missed by the IQR and z-score screens |
| Three users with 975–987 kcal | calories | Not a problem | **keep** | Flagged by the IQR fence only; consistent with their 198–199 sessions |

No value is physically impossible, so there are no critical anomalies. The two distances are not *provably*
wrong, which is why the record is kept and only the suspect cell is blanked. `04` repeats the main results
with both values kept as recorded and with both corrected by a factor of ten.

## Missing data

| Record | Missing | Decision | Evidence |
|---|---|---|---|
| User 4907 (Active, Rural, 155 sessions) | distance | **leave blank** | One value in 5,000. An imputed value would add nothing a model could use. Analyses that need distance use the 4,997 complete records and say so |
| User 5259 | sessions | **removed with the copy block** | An altered copy of user 3817 (location changed, sessions blanked) |

## Activity Level

| Field | Finding | Decision |
|---|---|---|
| *Activity Level* | Equals the band of *App Sessions* (50–99, 100–149, 150–199) in all 5,000 records | **keep as a derived label**; never use *App Sessions* to predict it (`04`), and do not read it as physical activity (`05`) |

In [10]:
from src.data.quality import quality_report
from src.visualization.presentation import save_table

_ = save_table(quality_report(raw, subset=FIELDS), '01_quality_raw')
_ = save_table(altered, '01_altered_copies')

PROFILE
rows: 5,299   columns: 8
memory: 1,064.4 KB

dtypes:
int64      3
str        3
float64    2

MISSING
complete rows: 5,297 of 5,299 (100.0%)
                         missing  missing_pct
App Sessions                   1         0.02
Distance Travelled (km)        1         0.02

DUPLICATES
exact duplicate rows: 0
duplicate on ['Gender', 'Age', 'Activity Level', 'Location', 'App Sessions', 'Distance Travelled (km)', 'Calories Burned']: 272

OUTLIERS (flagged, not removed)
Values flagged. Flagged is not the same as wrong -- decide and record why.
                 column    n  iqr_flagged  z_flagged  lower_fence  upper_fence   min    max
                User ID 5299            0          0     -2648.00      7948.00   1.0 5299.0
                    Age 5299            0          0        -3.50        80.50  18.0   59.0
           App Sessions 5298            0          0       -19.00       269.00  50.0  199.0
Distance Travelled (km) 5298            1          1       -19.50       20

## What these data can and cannot answer

**Can answer** (5,000 users, one year, no time detail):
- How app sessions, distance and calories relate to one another, and how tightly.
- Whether engagement differs by gender, age or location type, with enough users per group (about 1,600–2,600)
  to detect small differences.
- How well engagement can be predicted from the other fields, measured out of sample.
- How identifiable users are from the combination of their attributes.

**Cannot answer:**
- **Causation.** The records are observational; nothing was randomised.
- **Change over time.** No dates, so no seasonality, trend or churn.
- **Physical effort.** No duration or heart rate. Calories are relative values, not physiological totals.
- **App performance.** No crash, latency or error data.
- **Groups the file does not record:** users under 18 or over 59, gender beyond two categories, ethnicity,
  language, disability or income.

**Questions for the data owner:**
1. What produced the block after ID 5000: a re-run of an export, a sync retry, or test data?
2. What are the units and the period of *Calories Burned*?
3. Is *Activity Level* computed from sessions in the app's code? If so, it should be documented as a derived
  field.

## References

Kim, W., Choi, B.-J., Hong, E.-K., Kim, S.-K., & Lee, D. (2003). A taxonomy of dirty data. *Data Mining and
Knowledge Discovery, 7*(1), 81–99. https://doi.org/10.1023/A:1021564703268

Polyzotis, N., Zinkevich, M., Roy, S., Breck, E., & Whang, S. (2019). Data validation for machine learning.
*Proceedings of Machine Learning and Systems, 1*, 334–347.